In [29]:
import Pkg
Pkg.add("HiGHS")
Pkg.add("JuMP")

   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
Precompiling packages...
              ✗ CPLEX
  0 dependencies successfully precompiled in 2 seconds. 59 already precompiled.

The following 1 direct dependency failed to precompile:

CPLEX 

Failed to precompile CPLEX [a076750e-1247-5638-91d2-ce28b192dca0] to "/home/husted42/.julia/compiled/v1.12/CPLEX/jl_lAq3sZ".
ERROR: LoadError: CPLEX not properly installed. Please run Pkg.build("CPLEX")
Stacktrace:
  [1] error(s::String)
    @ Base ./error.jl:44
  [2] top-level scope
    @ ~/.julia/packages/CPLEX/5jmjD/src/CPLEX.jl:12
  [3] include(mod::Module, _path::String)
    @ Base ./Base.jl:306
  [4] include_package_for_output(pkg::Base.PkgId, input::String, depot_path::Vector{String}, dl_load_path::Vector{String}, load_path::Vector{String}, concrete_deps::Vector{Pair{Base.P

In [ ]:


using JuMP, HiGHS

########## ---------- Variables ---------- ##########
coords = [
    0 0 0;
    1 104 19;
    2 370 305;
    3 651 221;
    4 112 121;
    6 134 515
]

n_stops = size(coords, 1)

dist = zeros(Float64, n_stops, n_stops)

for i in 1:n_stops
    for j in 1:n_stops
        x1 = coords[i, 2]
        y1 = coords[i, 3]

        x2 = coords[j, 2]
        y2 = coords[j, 3]

        dist[i, j] = sqrt((x1 - x2)^2 + (y1 - y2)^2)
    end
end


########## ---------- Models ---------- ##########
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

########## ---------- Variables ---------- ##########
@variable(model, x[1:n_stops, 1:n_stops], Bin)
@variable(model, u[1:n_stops] >= 1)



########## ---------- Objectives ---------- ##########
@objective(model, Min, sum(x[i,j] * dist[i,j] for i in 1:n_stops, j in 1:n_stops))

########## ---------- Constraints ---------- ##########
@constraint(model, [i in 1:n_stops], x[i, i] == 0)
@constraint(model, [i in 1:n_stops], sum(x[i,j] for j in 1:n_stops) == 1)
@constraint(model, [j in 1:n_stops], sum(x[i,j] for i in 1:n_stops) == 1)

########## ---------- Subtour elimination (MTZ) ---------- ##########
# This is the order constraint
@constraint(model, u[1] == 1)
@constraint(model, [i in 2:n_stops], u[i] >= 2)
@constraint(model, [i in 2:n_stops], u[i] <= n_stops)

# For i != j and i,j in 2..n:
# u[i] - u[j] + n*x[i,j] <= n-1
@constraint(model, [i in 2:n_stops, j in 2:n_stops; i != j],
    u[i] - u[j] + n_stops * x[i, j] <= n_stops - 1
)

optimize!(model)
println("Optimal solution:")
println(objective_value(model))
println(value.(u))

for i in 1:n_stops
    println(value.(x[i, :]))
end

Optimal solution:
1857.5117454357478
[1.0, 5.999999999999993, 3.9999999999999964, 4.9999999999999964, 1.9999999999999996, 2.9999999999999982]
[0.0, -0.0, 0.0, 0.0, 0.9999999999999996, 0.0]
[1.0, 0.0, 0.0, -0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 1.0, 0.0, -0.0]
[0.0, 0.9999999999999996, -0.0, 0.0, -0.0, -0.0]
[0.0, -0.0, -0.0, -0.0, 0.0, 1.0]
[-0.0, 0.0, 1.0, 0.0, -0.0, 0.0]


# Tennis chours

In [ ]:
import Pkg
Pkg.add("HiGHS")
Pkg.add("JuMP")

using JuMP, HiGHS


In [ ]:
coords = [
    1   0.0   0.0;    # A0
    2   4.5   0.0;    # B0
    3  31.5   0.0;    # D0
    4  36.0   0.0;    # E0
    5   4.5  18.0;    # B1
    6  18.0  18.0;    # C1
    7  31.5  18.0;    # D1
    8   0.0  39.0;    # A2
    9   4.5  39.0;    # B2
   10  18.0  39.0;    # C2
   11  31.5  39.0;    # D2
   12  36.0  39.0     # E2
]

n_stops = size(coords)[1]
possible_routes = zeros(Int, 12, 12)

edges = [
    (1,2), (1,8),
    (2,5), (2,3),
    (3,4), (3,7),
    (4,12),
    (5,6), (5,9),
    (6,7), (6,10),
    (7,11)
]

for (i,j) in edges
    possible_routes[i,j] = 1
    possible_routes[j,i] = 1   # remove this line if you want directed
end


distance = zeros(Float64, n_stops, n_stops)

for i in 1:n_stops
    for j in 1:n_stops
        if possible_routes[i,j] == 1
            x1 = coords[i, 2]
            y1 = coords[i, 3]

            x2 = coords[j, 2]
            y2 = coords[j, 3]

            distance[i, j] = sqrt((x1 - x2)^2 + (y1 - y2)^2)
        end
    end
end



########## ---------- Models ---------- ##########
########## ---------- Models ---------- ##########
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

########## ---------- Variables ---------- ##########    possible_routes[j,i] = 1   # remove this line if you want directed

@variable(model, x[1:n_stops, 1:n_stops], Bin)



########## ---------- Objectives ---------- ##########
@objective(model, Min, sum(x[i,j] * distance[i,j] for i in 1:n_stops, j in 1:n_stops))

########## ---------- Constraints ---------- ##########
# Only allow traversal on permitted arcs
for i in 1:n_stops, j in 1:n_stops
    if possible_routes[i,j] == 0
        @constraint(model, x[i,j] == 0)
    end
end

# Flow conservation (what goes in must come out)
@constraint(model, [i in 1:n_stops],
    sum(x[i,j] for j in 1:n_stops) == sum(x[j,i] for j in 1:n_stops)
)

# Force sweeping of required undirected lines
@constraint(model, [i in 1:n_stops, j in 1:n_stops],
    x[i,j] + x[j,i] >= possible_routes[i,j]
)


optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))



Optimal solution:
z = 300.0
